In [9]:
import numpy as np
from scipy.linalg import eigh_tridiagonal

# ---------- CONSTANT ----------
HARTREE_TO_EV = 27.211386

# ---------- GRID ----------
Z = 2.0
r_max = 20.0
N = 2000

dr = r_max / N
r = np.linspace(dr, r_max, N)

# ---------- INITIAL GUESS ----------
u = 2 * r * np.exp(-Z * r)
u /= np.sqrt(np.trapezoid(u**2, r))

def hartree_potential(u):
    # Hartree-Fock'ta bir elektron diğer elektronu hisseder.
    # Bu yüzden yoğunlukta "2*" çarpanı olmamalıdır.
    rho_radial = u**2  # 4 * pi * r**2 * rho (1 elektron için)
    
    # İçeride kalan yük (Q)
    Q = np.cumsum(rho_radial) * dr
    term1 = Q / r

    # r'den sonsuza kadar olan integral (Vektörel hızlandırma)
    term2 = np.cumsum((rho_radial / r)[::-1])[::-1] * dr

    return term1 + term2

threshold = 1e-7
E_old = 0

for iteration in range(100):

    V_H = hartree_potential(u)
    V_eff = -Z/r + V_H

    # radyal kinetik operatör
    diag = 1/dr**2 + V_eff
    off = -0.5/dr**2 * np.ones(N-1)

    epsilon, vecs = eigh_tridiagonal(diag, off)

    u_new = vecs[:, 0]
    u_new /= np.sqrt(np.trapezoid(u_new**2, r))

    # Çift sayılmayı (double counting) önlemek için J enerjisi 1 kez çıkarılır.
    E_total = 2 * epsilon[0] - np.trapezoid(V_H * (u_new**2), r)

    print(f"{iteration+1:2d}. Iterasyon: {E_total * HARTREE_TO_EV:.4f} eV")

    if abs(E_total - E_old) < threshold:
        break

    u = u_new
    E_old = E_total

print("\nFinal HF Energy (Hartree):", E_total)
print("Final HF Energy (eV):", E_total * HARTREE_TO_EV)

 1. Iterasyon: -73.7785 eV
 2. Iterasyon: -78.8278 eV
 3. Iterasyon: -77.2156 eV
 4. Iterasyon: -77.6990 eV
 5. Iterasyon: -77.5508 eV
 6. Iterasyon: -77.5959 eV
 7. Iterasyon: -77.5821 eV
 8. Iterasyon: -77.5864 eV
 9. Iterasyon: -77.5851 eV
10. Iterasyon: -77.5855 eV
11. Iterasyon: -77.5853 eV
12. Iterasyon: -77.5854 eV
13. Iterasyon: -77.5854 eV
14. Iterasyon: -77.5854 eV
15. Iterasyon: -77.5854 eV

Final HF Energy (Hartree): -2.8512097466977107
Final HF Energy (eV): -77.58536898435364
